# INFO 159/259
#<center> Homework 6: Unsupervised Methods </center>

<center> Due: April 30, 2026 @ 11:59pm </center>


In class, we have covered several unsuperivsed methods for working with data. In this final homework, we want you to tell us something interesting about a dataset with any of the unsuperivised methods we covered during class.

## Dataset
You can pick any of the datasets below for your analysis:

 <a name="cell-id"></a>
    [CMU Book Summary Dataset](https://www.cs.cmu.edu/~dbamman/booksummaries.html)
        Metadata: Title, author, publication date, genre

 <a name="cell-id"></a>
    [CMU Movie Summary Dataset](http://www.cs.cmu.edu/~ark/personas/)
        Metadata: Movie box office revenue, genre, release date, runtime, and language

 <a name="cell-id"></a>
    [Large Movie Review Dataset](https://ai.stanford.edu/~amaas/data/sentiment/)
        Metadata: positive/negative sentiment

 <a name="cell-id"></a>
    [Congressional Speech Data](https://www.cs.cornell.edu/home/llee/data/convote.html)
        Metadata: political party

 <a name="cell-id"></a>
    [100K ArXiv abstracts](https://drive.google.com/file/d/1ThK1D9AstYI6s2Z7m9SmvLqLZPneMp12/view?usp=sharing)
        Metadata: ArXiv subject (e.g., math, physics, cs)


## Deliverable 1: Tell us about the dataset you picked.
In the text box below, tell us which dataset you selected and why you are interested in it. Describe the metadata of your selected dataset.

(your response goes here)

## Deliverable 2: Tell us about the methods you chose


For this assignment, you will use two methods from class to explore your data and tell us something interesting about it.

a.) You must use one of the following methods:

* Topic modeling
* Hierarchical clustering
* K-means clustering

b.) In addition, you should use one more method from the following options:

* PMI
* dependency parsing
* NER
* WSD (using WordNet)
* Relation extration
* Classification (e.g. logistic regression, BERT)
* Word embeddings (word2vec, contextual embeddings)

You are free to use any external resources and libraries for this assignment; the goal is to put use your comprehensive knowledge of NLP to use.

As starting points, you can use `spacy` for parsing/NER, `lda` for topic modeling, `sklearn` for clustering (KMeans, TF-IDF). You can implement methods for which you cannot find libraries.

In the text box below, tell us about:
* which unsupervised method you chose and why you chose that method for this particular dataset (1 sentence)
* what pre-processing steps you had to perform to run the method on your data (up to 2 sentences)
* how you choose to represent your documents if using clustering methods (up to 2 sentences)


## Deliverable 3: Code for analysis
In the code block below, add your code for analysis. You can organize your code in way that fits best for your analysis, including adding more code blocks.

In [76]:
#read/clean movie metadata text file and make dictionaries
#numeric id is clean six digits
#freebase id has slashes and letters
title_to_numeric = {}
freebase_to_numeric = {}
with open("movie.metadata.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        numeric_id = parts[0]
        freebase_id = parts[1]
        title = parts[2]
        title_to_numeric[title] = numeric_id #title: 123456
        freebase_to_numeric[freebase_id] = numeric_id #m/0ab1c3: 123456

In [70]:
#read/clean tv tropes text file and make dictionary structured like trope: movie titles
import json
trope_mov_pairs = {} #empty dictonary to be structured as trope: movie1, movie2, etc.
with open("tvtropes.clusters.txt", "r", encoding="utf-8") as f:
  for line in f:
    line = line.strip() #clean newline
    parts = line.split("\t") #clean tabs and convert into list
    trope = parts[0] #index 0 is the trope
    data = json.loads(parts[1]) #text into dictionary (keys are character, movie, id, actor)
    title = data["movie"]
    if trope in trope_mov_pairs:
      if title not in trope_mov_pairs[trope]:
        trope_mov_pairs[trope].append(title)
    else: trope_mov_pairs[trope] = [title]
print(trope_mov_pairs)

{'absent_minded_professor': ['Flubber', 'Richie Rich', 'The Shadow', 'Them!', 'Stargate'], 'adventurer_archaeologist': ['Indiana Jones and the Kingdom of the Crystal Skull', 'Indiana Jones and the Raiders of the Lost Ark', 'Indiana Jones and the Temple of Doom', 'The Mummy'], 'arrogant_kungfu_guy': ['Enter the Dragon', 'The Karate Kid', 'Crouching Tiger, Hidden Dragon', 'Kill Bill Volume 2', 'Rocky', 'Rocky III', 'G.I. Joe: The Rise of Cobra', 'Highlander', 'Star Wars Episode III: Revenge of the Sith'], 'big_man_on_campus': ['Never Been Kissed', 'John Tucker Must Die', "It's a Boy Girl Thing", "National Lampoon's Van Wilder", 'Election', 'High School Musical', "Can't Hardly Wait"], 'bounty_hunter': ['For a Few Dollars More', 'The Outlaw Josey Wales', 'The Proposition', 'The Rundown', 'Blade Runner', 'Raising Arizona', 'Midnight Run', 'The Bounty Hunter', 'The Chronicles of Riddick', 'Wanted: Dead or Alive'], 'brainless_beauty': ["Fool's Gold", 'The Opposite of Sex', 'Election', 'The Ho

In [79]:
#read/clean plot summaries text file and make dictioanry structured like id: summary
movie_plots = {}
with open("plot_summaries.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t", 1)
        id, plot = parts
        movie_plots[id] = plot

In [101]:
#building documents
#have to use previous dictionaries to convert between the different ids
docs = {}
for trope, titles in trope_mov_pairs.items():
    summaries = []
    for title in titles:
        if title in title_to_numeric:
            numeric_id = title_to_numeric[title]
        if numeric_id is None:
            continue
        plot = movie_plots.get(numeric_id)
        if plot:
            summaries.append(plot)
    docs[trope] = " ".join(summaries)

In [107]:
#TF-IDF: each trope doc becomes vector of weights
from sklearn.feature_extraction.text import TfidfVectorizer
corpus = list(docs.values()) #plot summaries
labels = list(docs.keys()) #tropes
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
X = vectorizer.fit_transform(corpus)

In [108]:
import numpy as np
feature_names = vectorizer.get_feature_names_out()
for i, trope in enumerate(labels):
    row = X[i].toarray().flatten() #each row i in X corresponds to a trope
    top_indices = row.argsort()[-10:][::-1] #lowest to highest top 10 TF-IDF scores
    #then reverse order to top 10 sorted highest to lowest
    top_words = [feature_names[j] for j in top_indices]
    print(trope, ":", top_words)

absent_minded_professor : ['richie', 'philip', 'rich', 'dough', 'flubber', 'van', 'cadbury', 'ferguson', 'weebo', 'keenbean']
adventurer_archaeologist : ['jones', 'indiana', 'soviets', 'mutt', 'oxley', 'skull', 'stones', 'marion', 'mola', 'round']
arrogant_kungfu_guy : ['jen', 'rocky', 'mu', 'bai', 'lien', 'shu', 'creed', 'fox', 'lo', 'adrian']
big_man_on_campus : ['woody', 'nell', 'yale', 'nicky', 'school', 'interview', 'dance', 'day', 'museum', 'homecoming']
bounty_hunter : ['riddick', 'marshal', 'lord', 'vaako', 'toombs', 'necromongers', 'kyra', 'imam', 'necromonger', 'purifier']
brainless_beauty : ['maggie', 'dedee', 'matt', 'rose', 'ella', 'randy', 'lucia', 'ashes', 'grandmother', 'job']
broken_bird : ['nina', 'chelsea', 'logan', 'lily', 'taft', 'swan', 'kelly', 'thomas', 'attorney', 'painting']
bromantic_foil : ['000', 'zorin', 'zone', 'zombies', 'zombie', 'zolm', 'zinj', 'zeke', '2009', '209']
bruiser_with_a_soft_center : ['bond', 'flynn', 'chiffre', 'mcp', 'le', 'sark', 'tron',

In [118]:
#topic clustering (NMF: discover patterns)
from sklearn.decomposition import NMF
nmf = NMF(n_components=10, random_state=42)
W = nmf.fit_transform(X) #number of docs by 10 topics
H = nmf.components_  #each row is a topic and each value is importance of word
#multiply W and H to get X

In [119]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H): #loop through topic
    top_indices = topic.argsort()[-10:][::-1] #indices of top 10 most important words per topic
    top_words = [feature_names[i] for i in top_indices] #convert indices into words
    print(f"Topic {topic_idx}:", top_words) #each topic is a cluster of words that tend to appear together

Topic 0: ['bond', 'chiffre', 'le', 'vesper', 'tournament', 'white', 'winnings', 'dimitrios', 'agent', 'miami']
Topic 1: ['luke', 'han', 'vader', 'leia', 'lando', 'falcon', 'obi', 'wan', 'millennium', 'fett']
Topic 2: ['stark', 'rochester', 'stane', 'batman', 'dent', 'joker', 'reactor', 'yinsen', 'rings', 'suit']
Topic 3: ['nick', 'demarco', 'holly', 'mr', 'kira', 'hanna', 'jerry', 'cassie', 'wiegler', 'carver']
Topic 4: ['jen', 'willard', 'bai', 'mu', 'kurtz', 'lien', 'shu', 'fox', 'horton', 'lo']
Topic 5: ['kumar', 'harold', 'alison', 'marijuana', 'ben', 'fabious', 'thadeous', 'leezar', 'castle', 'sandy']
Topic 6: ['belle', 'beast', 'veronica', 'heather', 'avenant', 'jimmy', 'castle', 'ludovic', 'ellie', 'father']
Topic 7: ['tony', 'frank', 'manny', 'sosa', 'bobby', 'gina', 'jake', 'elvira', 'anna', 'chi']
Topic 8: ['woody', 'wade', 'nell', 'reisman', 'jones', 'ryan', 'buzz', 'ronnie', 'munny', 'evans']
Topic 9: ['flynn', 'mcp', 'sark', 'tron', 'dillinger', 'encom', 'programs', 'core'

In [121]:
for i, trope in enumerate(labels): #loop through each trope
    top_topic = W[i].argmax() #index of largest value  to get most dominant topic for trope
    print(trope, "→ Topic", top_topic)

absent_minded_professor → Topic 8
adventurer_archaeologist → Topic 8
arrogant_kungfu_guy → Topic 4
big_man_on_campus → Topic 8
bounty_hunter → Topic 8
brainless_beauty → Topic 8
broken_bird → Topic 8
bromantic_foil → Topic 0
bruiser_with_a_soft_center → Topic 9
bully → Topic 0
byronic_hero → Topic 2
casanova → Topic 2
chanteuse → Topic 8
charmer → Topic 0
child_prodigy → Topic 0
classy_cat_burglar → Topic 0
consummate_professional → Topic 0
corrupt_corporate_executive → Topic 9
coward → Topic 8
crazy_jealous_guy → Topic 7
crazy_survivalist → Topic 0
cultured_badass → Topic 0
dean_bitterman → Topic 0
dirty_cop → Topic 8
ditz → Topic 8
doormat → Topic 0
drill_sargeant_nasty → Topic 8
dumb_blonde → Topic 8
dumb_muscle → Topic 0
eccentric_mentor → Topic 1
egomaniac_hunter → Topic 6
evil_prince → Topic 8
fastest_gun_in_the_west → Topic 8
father_to_his_men → Topic 4
final_girl → Topic 8
gadgeteer_genius → Topic 2
gentleman_thief → Topic 3
granola_person → Topic 8
grumpy_old_man → Topic 8
har

In [124]:
#format/organize topics to show first 10 tropes per topic
from collections import defaultdict
topic_groups = defaultdict(list)
for i, trope in enumerate(labels):
    topic = W[i].argmax()
    topic_groups[topic].append(trope)
for topic, tropes in topic_groups.items():
    print(f"\nTopic {topic}:")
    print(tropes[:10])


Topic 8:
['absent_minded_professor', 'adventurer_archaeologist', 'big_man_on_campus', 'bounty_hunter', 'brainless_beauty', 'broken_bird', 'chanteuse', 'coward', 'dirty_cop', 'ditz']

Topic 4:
['arrogant_kungfu_guy', 'father_to_his_men', 'surfer_dude', 'warrior_poet']

Topic 0:
['bromantic_foil', 'bully', 'charmer', 'child_prodigy', 'classy_cat_burglar', 'consummate_professional', 'crazy_survivalist', 'cultured_badass', 'dean_bitterman', 'doormat']

Topic 9:
['bruiser_with_a_soft_center', 'corrupt_corporate_executive', 'playful_hacker']

Topic 2:
['byronic_hero', 'casanova', 'gadgeteer_genius', 'heartbroken_badass']

Topic 7:
['crazy_jealous_guy', 'ophelia']

Topic 1:
['eccentric_mentor', 'loveable_rogue', 'master_swordsman', 'pupil_turned_to_evil', 'trickster']

Topic 6:
['egomaniac_hunter', 'jerk_jock']

Topic 3:
['gentleman_thief', 'junkie_prophet', 'psycho_for_hire', 'stupid_crooks']

Topic 5:
['loser_protagonist', 'slacker', 'stoner']


## Deliverable 4: Results and Analysis
Here, use code or text blocks below to reflect on interesting results from your analysis. You can use graphs or figures from your code above.

Additionally, **add your interpretation of the results**. What did you find interesting from what you obsereved?

# Submission
Congratulations on finishing HW6! Please ensure that you submit this completed notebook onto Gradescope. Make sure all cells in the notebook are run so that print statements are visible.

`File` --> `Download` --> `Download .ipynb`

Make sure your notebook is named `HW6.ipynb` when you upload it to [Gradescope](https://www.gradescope.com/courses/1238346/) before April 30, 2026 at 11:59pm.